# Sanity 05 — the results, both tables

Mirrors `scripts/05_train_downstream.py` + `scripts/bootstrap_ci.py` + `scripts/paired_bootstrap.py`.
The campaign uses **two evaluation sets with different jobs** (never compare numbers across them):

1. **Their protocol** (`downstream/<scale>/`) — the 100k stratified test with 112 frauds. The only
   set comparable to NVIDIA's published points (baseline 0.1238, fusion 0.1755). Wide CIs by nature.
2. **Full test period** (`downstream/<scale>_fulltest/`) — all 2.44M test rows, 2,724 frauds; ~5x
   tighter CIs; used for our model-vs-model claims and the context-ordering result.

Environment for all persisted metrics: xgboost==3.2.0 (their pin), XGBoost on CUDA (their device).
Pre-pin metrics were set aside as `benchmark_metrics_prepin.json`.

In [ ]:
import os, json
import numpy as np
import pandas as pd

BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")

In [ ]:
# Table 1 — their protocol (100k), with bootstrap CIs and P(beats their fusion).
rows = []
for sc in ("full", "xl", "xxl"):
    p = f"{BASE}/downstream/{sc}/bootstrap_ci.json"
    if not os.path.exists(p): continue
    ci = json.load(open(p))
    for m, r in ci["results"].items():
        rows.append({"scale": sc, "model": m, "ap": round(r["ap"], 4),
                     "ci": [round(x, 3) for x in r["ap_ci95"]],
                     "P(>0.1755)": round(r["p_beats_nvidia_fusion"], 3)})
pd.DataFrame(rows)

**What to check in table 1:** every `embed_xgb` row has P(>0.1755) at 0.94+; the *baseline*
row has P ≈ 0.16 — i.e. with only 112 test frauds even the raw baseline sometimes clears their
fusion bar in a bootstrap draw. That is why no ordering among context lengths is claimed from
this table. (Also note the xxl row here is the 40-epoch continuation model.)

In [ ]:
# Table 2 — full test period (2.44M rows), the tight internal comparison.
rows = []
for sc in ("full", "xl", "xxl"):
    p = f"{BASE}/downstream/{sc}_fulltest/bootstrap_ci.json"
    if not os.path.exists(p): continue
    ci = json.load(open(p))
    for m, r in ci["results"].items():
        rows.append({"scale": sc, "model": m, "ap": round(r["ap"], 4),
                     "ci": [round(x, 3) for x in r["ap_ci95"]]})
t2 = pd.DataFrame(rows).pivot(index="model", columns="scale", values="ap")
t2

In [ ]:
# The ordering claim, done right: PAIRED bootstrap (same resampled rows for every model).
pb = json.load(open(f"{BASE}/downstream/paired_bootstrap_embed_xgb.json"))
print("points:", {k: round(v, 4) for k, v in pb["point"].items()})
for pair, r in pb["pairs"].items():
    print(f"{pair:<34} diff {r['mean_diff']:+.4f}  CI {np.round(r['ci95'], 4)}  P(A>B) {r['p_a_gt_b']:.3f}")

**What to check in table 2 + the pairing:** baseline 0.2081; embed_xgb 0.2788 / 0.3027 / 0.2665
for 512/1024/2048 — a strict, all-significant ordering **1024 > 512 > 2048**. The 2048 model had
DOUBLE the training budget (40 epochs) and still lost to 1024 → the fall-off is burst-structure
dilution, not undertraining. Two corroborations visible in the tables: the PCA64 row is flat
(~0.20) at 512/1024 and *catches up* to the full embedding at 2048 (~0.27 ≈ 0.267 — little signal
beyond 64 components left by then); and the linear head collapses with context (0.213 → 0.121 →
0.167) while trees stay strong.

Controls live alongside: `downstream/full_probe/probe_metrics_shuffled_seed0.json` (shuffled-label
sanity — AP at the prevalence floor) and `downstream/full_velocity/velocity_metrics.json` (velocity
features don't recover the lift). Raw per-model test predictions for any further resampling:
`downstream/<run>/test_predictions.parquet`.